In [9]:
from langchain_ollama import ChatOllama

class QuestionNode:
    def __init__(self):
      print("Creating LLM") 
      self.llm = ChatOllama(model="llama3.2:1b")

    def __call__(self, state):
        response = self.llm.invoke(state["question"])
        return{
            "answer" : response.content
        }

node = QuestionNode()
builder.add_node("question",node)
      

Creating LLM


NameError: name 'builder' is not defined

In [7]:
!ollama list

]11;?\NAME                ID              SIZE      MODIFIED     
deepseek-r1:1.5b    e0979632db5a    1.1 GB    26 hours ago    
llama3.2:1b         baf6a787fdff    1.3 GB    10 days ago     


In [ ]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END


# -----------------------------
# 1. STATE
# -----------------------------
# State means the data that travels through the graph.
# Every node receives this same state object.
class SupportState(BaseModel):
    query: str                         # User question
    category: str = ""                 # Router/classifier decision
    answer: str = ""                   # Final answer
    nodes_ran: list[str] = Field(default_factory=list)  # Track executed nodes


# -----------------------------
# 2. CLASSIFIER NODE
# -----------------------------
# This node decides what type of query it is.
class ClassifierNode:
    def __init__(self):
        # __init__ runs only once when object is created.
        print("ClassifierNode object created")

    def __call__(self, state: SupportState):
        # __call__ runs when LangGraph executes this node.
        state.nodes_ran.append("classifier")

        # Simple logic instead of LLM for now.
        if "refund" in state.query.lower():
            state.category = "refund"
        elif "password" in state.query.lower():
            state.category = "technical"
        else:
            state.category = "general"

        return state


# -----------------------------
# 3. REFUND NODE
# -----------------------------
class RefundNode:
    def __call__(self, state: SupportState):
        state.nodes_ran.append("refund_node")
        state.answer = "Your refund request will be checked by our billing team."
        return state


# -----------------------------
# 4. TECHNICAL NODE
# -----------------------------
class TechnicalNode:
    def __call__(self, state: SupportState):
        state.nodes_ran.append("technical_node")
        state.answer = "Please reset your password using the Forgot Password option."
        return state


# -----------------------------
# 5. GENERAL NODE
# -----------------------------
class GeneralNode:
    def __call__(self, state: SupportState):
        state.nodes_ran.append("general_node")
        state.answer = "Our support team will help you shortly."
        return state


# -----------------------------
# 6. EXTRA NODE 1
# -----------------------------
# This node is registered, but may not run.
class ManagerNode:
    def __call__(self, state: SupportState):
        state.nodes_ran.append("manager_node")
        return state


# -----------------------------
# 7. EXTRA NODE 2
# -----------------------------
# This node is also registered, but may not run.
class FeedbackNode:
    def __call__(self, state: SupportState):
        state.nodes_ran.append("feedback_node")
        return state


# -----------------------------
# 8. ROUTER FUNCTION
# -----------------------------
# Router decides which node should run next.
# Router does not return state.
# Router returns node name.
def route_query(state: SupportState):
    if state.category == "refund":
        return "refund"

    elif state.category == "technical":
        return "technical"

    else:
        return "general"


# -----------------------------
# 9. BUILD GRAPH
# -----------------------------
builder = StateGraph(SupportState)

# Register all nodes.
# Registration does not mean execution.
builder.add_node("classifier", ClassifierNode())
builder.add_node("refund", RefundNode())
builder.add_node("technical", TechnicalNode())
builder.add_node("general", GeneralNode())

# Extra nodes are registered but not connected.
builder.add_node("manager", ManagerNode())
builder.add_node("feedback", FeedbackNode())


# -----------------------------
# 10. DEFINE FLOW
# -----------------------------
# First node to run
builder.set_entry_point("classifier")

# After classifier, router decides next node.
builder.add_conditional_edges(
    "classifier",
    route_query,
    {
        "refund": "refund",
        "technical": "technical",
        "general": "general"
    }
)

# After selected node, graph ends.
builder.add_edge("refund", END)
builder.add_edge("technical", END)
builder.add_edge("general", END)


# -----------------------------
# 11. COMPILE GRAPH
# -----------------------------
graph = builder.compile()


# -----------------------------
# 12. RUN GRAPH
# -----------------------------
result = graph.invoke(
    SupportState(
        query="I forgot my password"
    )
)

print(result)
print(result["nodes_ran"])
print(result["answer"])